# Phase 3 — GDELT GKG Post-Processing Pipeline (Colab)

**Purpose**: Process the 5.1 GB of raw GKG data through dedup → classify → daily aggregation, all on Colab's free RAM (12.7 GB) with no risk of OOM on your local machine.

**Setup**: Drive folder `WarSignalsThesis_Data/` (ID: `1i1kkelDYszQ5Bi5Hv94NGT6wjCHkbIWU`) is already populated with 184 raw parquet files.

**Outputs**: Saved back to `WarSignalsThesis_Data/data/processed/news/` on Drive.

## Cell 1: Mount Drive and Clone Repo

In [ ]:
import os
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/WarSignalsThesis_Data')
REPO_DIR = Path('/content/WarSignalsThesis')

# Clone repo if not already cloned
if not REPO_DIR.exists():
    !git clone https://github.com/katerynavalenia/WarSignalsThesis

os.chdir(REPO_DIR)
print(f'✓ Working directory: {os.getcwd()}')
print(f'✓ Drive mounted: {DRIVE_ROOT.exists()}')
print(f'✓ Raw data files: {len(list((DRIVE_ROOT / "data/raw_enriched").glob("*.parquet")))}')

## Cell 2: Install Dependencies

In [ ]:
!pip install -q pandas pyarrow tqdm pyyaml datasketch
print('✓ Dependencies installed')

## Cell 3: Symlink Local Data Dirs to Drive

In [ ]:
# Create symlinks so the pipeline can use local paths but data lives on Drive
for subdir in ['raw_enriched']:
    src = DRIVE_ROOT / 'data' / subdir
    dst = REPO_DIR / 'data' / 'news_colab_sim' / 'war_signals_phase3' / subdir
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        dst.symlink_to(src)
        print(f'✓ Symlinked {dst} → {src}')
    else:
        print(f'✓ Already exists: {dst}')

# Ensure processed dirs exist
for subdir in ['processed/news', 'interim']:
    p = REPO_ROOT / 'data' / subdir if False else DRIVE_ROOT / 'data' / subdir
    p.mkdir(parents=True, exist_ok=True)
    print(f'✓ Ensured dir: {p}')

## Cell 4: Run the Pipeline (Chunked, Memory-Safe)

In [ ]:
import sys
import time

sys.path.insert(0, '/content/WarSignalsThesis')

t0 = time.time()
!python scripts/phase3_post_process_enriched.py 2>&1 | tee /tmp/pipeline.log
elapsed = time.time() - t0
print(f'\n✓ Pipeline completed in {elapsed/60:.1f} minutes')

## Cell 5: Verify Outputs

In [ ]:
import pandas as pd
from pathlib import Path

OUT = DRIVE_ROOT / 'data' / 'processed' / 'news'

print('=== OUTPUT FILES ===')
for f in sorted(OUT.iterdir()):
    if f.is_file():
        size_mb = f.stat().st_size / 1024 / 1024
        print(f'  {f.name:50s}  {size_mb:8.1f} MB')

print('\n=== DAILY AGGREGATE ===')
daily = pd.read_parquet(OUT / 'news_daily_enriched.parquet')
print(f'Shape: {daily.shape}')
print(f'Columns: {list(daily.columns)}')
print(f'Date range: {daily.index.min().date()} → {daily.index.max().date()}')
print(f'Total articles: {daily["n_articles_total"].sum():,}')
print(f'Mean articles/day: {daily["n_articles_total"].mean():.0f}')

print('\n=== SOURCE GROUP DISTRIBUTION (full dataset) ===')
classified = pd.read_parquet(OUT / 'gdelt_articles_classified_enriched.parquet')
vc = classified['source_group'].value_counts()
for group, count in vc.items():
    pct = count / len(classified) * 100
    print(f'  {group:15s}  {count:>10,}  ({pct:5.1f}%)')

print('\n=== TONE BY SOURCE GROUP ===')
classified['date'] = pd.to_datetime(classified['date'], errors='coerce')
valid = classified.dropna(subset=['date'])
for group in ['ukrainian', 'russian', 'western', 'other']:
    sub = valid[valid['source_group'] == group]
    if not sub.empty and 'tone_avg' in sub.columns:
        tones = sub['tone_avg'].dropna()
        if not tones.empty:
            print(f'  {group:15s}  mean: {tones.mean():>7.2f}  median: {tones.median():>7.2f}  n: {len(tones):>8,}')

## Cell 6: Run Sensitivity Analysis (Optional, takes ~1 min)

In [ ]:
!python scripts/phase3_sensitivity_analysis.py 2>&1 | tail -20

## Cell 7: Final Check — All Outputs on Drive

In [ ]:
print('=== DRIVE FOLDER STRUCTURE ===')
!rclone lsf gdrive:WarSignalsThesis_Data/data/processed/news/
print()
print('=== TOTAL SIZE OF PROCESSED DATA ===')
!rclone size gdrive:WarSignalsThesis_Data/data/processed/news/